# Implementación de modelos supervisados (clasificación/regresión) con Scikit-learn

En este estudio se seleccionaron dos variables como target para los modelos supervisados, en función de su relevancia clínica y del tipo de problema que representan:

## Clasificación para Antecedentes Personales de Cáncer (Sí/No)
Antecedentes personales de cáncer (clasificación binaria):

Se eligió esta variable porque responde a una pregunta clínica clave: ¿el paciente ha tenido o no un cáncer previamente? Esta información es categórica (Sí/No), lo que la convierte en un problema de clasificación supervisada. Su análisis permite identificar factores asociados a la presencia de antecedentes oncológicos y construir modelos predictivos que apoyen la evaluación de riesgo en nuevos pacientes.

Variable objetivo utilizada: Antecedentes personales de cáncer — columna binaria generada a partir de la respuesta original del paciente (mapea 'Sí' → 1; 'No' → 0).


In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.data_preprocessing import load_data, encode_categoricals, split_and_scale_classification
from src.model_training import train_and_predict_classifiers
from src.model_evaluation import classification_metrics
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# 1. CARGAR DATOS
df = load_data("dataset_chile_cancer_piel.csv")

# 2. DEFINIR LA ETIQUETA (TARGET)
# Transformamos 'Sí' a 1 y 'No' a 0 para que el modelo lo entienda.
y = df['Antecedentes personales de cáncer'].map({'Sí': 1, 'No': 0})

# 3. DEFINIR LAS CARACTERÍSTICAS (X)
# Quitamos la columna que estamos prediciendo (Antecedentes personales de cáncer)
# Y también quitamos 'Cáncer familiar 1er grado (tipo)' porque si alguien tiene 
# un tipo anotado ahí, el modelo sabría la respuesta de antemano (sería trampa).
X = df.drop(['Antecedentes personales de cáncer', 'Cáncer familiar 1er grado (tipo)'], axis=1)

# 4. PREPROCESAMIENTO
# Convertimos el resto de textos (Sexo, Tabaquismo, Localización, etc.) a números
X = encode_categoricals(X, drop_first=True)

# 5. DIVISIÓN Y ESCALADO
# Usamos stratify=True para que en el grupo de prueba haya la misma proporción de 
# gente con antecedentes que en el de entrenamiento.
X_train_scaled, X_test_scaled, y_train, y_test, scaler = split_and_scale_classification(
    X, y, test_size=0.2, random_state=42, stratify=True
)

# 6. MODELOS
models = {
    "Regresion_Logistica": LogisticRegression(max_iter=1000, random_state=42),
    "Bosque_Aleatorio": RandomForestClassifier(random_state=42),
}

# 7. ENTRENAMIENTO Y PREDICCIÓN
trained, preds = train_and_predict_classifiers(models, X_train_scaled, X_test_scaled, y_train)

# 8. RESULTADOS
for name, y_pred in preds.items():
    metrics = classification_metrics(y_test, y_pred)
    print(f'--- Resultados para: {name} ---')
    print(metrics)

--- Resultados para: Regresion_Logistica ---
{'accuracy': 0.625, 'precision': 0.23076923076923078, 'recall': 0.04411764705882353, 'f1': 0.07407407407407407}
--- Resultados para: Bosque_Aleatorio ---
{'accuracy': 0.635, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}


##Interpretación y análisis

Logistic Regression
Accuracy = 0.625: El modelo acierta en un 62.5% de los casos, desempeño moderado.

Precision = 0.23: De las predicciones positivas, solo un 23% fueron correctas. Alto número de falsos positivos.

Recall = 0.04: Apenas detecta el 4% de los casos positivos reales. La mayoría de los pacientes con antecedentes no fueron identificados.

F1 = 0.07: El equilibrio entre precisión y sensibilidad es muy bajo.

Interpretación: Logistic Regression logra cierta clasificación correcta, pero su baja sensibilidad lo hace poco útil en un contexto clínico donde es crítico detectar todos los positivos.

Random Forest
Accuracy = 0.635: Acierta en un 63.5% de los casos.

Precision = 0.0 / Recall = 0.0 / F1 = 0.0: No detecta ningún caso positivo. Clasifica todos los pacientes como negativos.

Interpretación: Random Forest falla completamente en identificar la clase positiva. Su exactitud aparente se debe a que la mayoría de los pacientes no tenían antecedentes, y el modelo predijo siempre “No”.

## Regresión para Tamaño máximo (cm).

Tamaño máximo de la lesión (regresión):  
Se seleccionó porque es un indicador cuantitativo clave en la evolución clínica. Al ser un valor continuo (expresado en centímetros), corresponde a un problema de regresión supervisada. Modelar esta variable permite estimar el tamaño esperado de una lesión en función de características del paciente y factores de riesgo.

In [2]:
# Uso de utilidades desde src/ para regresión
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.data_preprocessing import load_data, encode_categoricals, split_and_scale_regression
from src.model_training import train_and_predict_classifiers
from src.model_evaluation import regression_metrics
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

y_reg = df["Tamaño máximo (cm)"]
X_reg = df.drop(["Tamaño máximo (cm)"], axis=1)
X_reg = encode_categoricals(X_reg, drop_first=True)

# División y escalado
X_train_r_scaled, X_test_r_scaled, y_train_r, y_test_r, scaler_r = split_and_scale_regression(X_reg, y_reg, test_size=0.2, random_state=42)

# Modelos
lin_reg = LinearRegression()
rf_reg = RandomForestRegressor(random_state=42)
models = {"LinearRegression": lin_reg, "RandomForestRegressor": rf_reg}

# Entrenamiento manual para regresores
for name, model in models.items():
    model.fit(X_train_r_scaled, y_train_r)
    y_pred = model.predict(X_test_r_scaled)
    print(f'--- {name} ---')
    print(regression_metrics(y_test_r, y_pred))


--- LinearRegression ---
{'mae': 1.5730349551594276, 'mse': 3.3099087761447676, 'rmse': 1.8193154691105025, 'r2': -0.023554707459312096}
--- RandomForestRegressor ---
{'mae': 1.6122000000000003, 'mse': 3.3924099000000005, 'rmse': 1.8418495866926812, 'r2': -0.049067318048859665}


## Interpretación y Analisis 
*Linear Regression*


El MAE de 1.57 cm indica que, en promedio, el modelo se equivoca alrededor de 1.5 cm al estimar el tamaño de la lesión.

El MSE de 3.31 muestra que existen errores más grandes en algunos casos, ya que penaliza más las desviaciones altas.

El R² negativo (-0.02) significa que el modelo no logra explicar la variabilidad de los datos; de hecho, predecir con este modelo es peor que usar simplemente la media del tamaño de las lesiones como referencia.


*Random Forest Regressor*

El MAE de 1.61 cm es muy similar al de la regresión lineal, lo que indica un error promedio comparable.

El MSE de 3.41 también refleja errores considerables en algunos casos.

El R² negativo (-0.05) confirma que el modelo tampoco logra capturar la relación entre las variables predictoras y el tamaño de la lesión, mostrando un desempeño inferior a una predicción basada en la media.